# Lab - Structured Output with Tool Use

One messy customer message, extracted two different ways.

| Task | What you learn |
| --- | --- |
| 1 | Asking for JSON in prose gives you whatever the model felt like |
| 2 | A tool schema gives you the fields you asked for, every time |
| 3 | Why that matters: your code can branch on it |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

The schema in Task 2 is long. You do not type it - it is provided, and the comments
point at the four design choices worth understanding.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import json
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()

model = "claude-sonnet-4-6"

# The messy message we want to turn into data. A human understands it
# immediately. An application cannot do anything with it as it stands.
CUSTOMER_MESSAGE = """
I got my shoes yesterday and they are scratched.
I want a replacement. I can send a photo if needed.
"""

# What our backend needs before it can act on a return.
REQUIRED_FIELDS = [
    "order_id",
    "item",
    "reason",
    "reason_detail",
    "desired_action",
    "desired_action_detail",
    "evidence_provided",
    "urgency",
    "missing_information",
]

---

## Task 1 - Ask for JSON without a schema

In [ ]:
# ============================================================
# TASK 1 - Ask for JSON without a schema
# ============================================================
#
# WHAT TO DO
#   Ask Claude, in plain words, to extract the return request
#   and return only valid JSON. Then look at what came back.
#
# WHY IT MATTERS
#   This is what most people try first, and it usually works,
#   which is the trap. Read the output carefully. It parsed -
#   fine. But which fields did you get? What are they called?
#   Is "damaged" the same as "damaged_item" to your router?
#
#   You asked for structure in a sentence, so the structure is
#   whatever the model decided. Nothing in the request said
#   otherwise.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Let's first look at the less reliable version" - about
#   1 minute 23 seconds in.
#
# HOW TO DO IT
#   One blank. Send JSON_PROMPT as a single user message, with
#   no tools at all - the tool is the next task.
# ============================================================

JSON_PROMPT = f"""
Extract the return request from this customer message.
Return only valid JSON.

Customer message:
{CUSTOMER_MESSAGE}
"""

# BEGIN SOLUTION
freeform = client.messages.create(
    model=model,
    max_tokens=500,
    messages=[
        {"role": "user", "content": JSON_PROMPT},
    ],
)
# SCAFFOLD: freeform = client.messages.create(
# SCAFFOLD:     model=model,
# SCAFFOLD:     max_tokens=500,
# SCAFFOLD:     messages=[
# SCAFFOLD:         {"role": "user", "content": ...},   # the JSON_PROMPT variable
# SCAFFOLD:     ],
# SCAFFOLD: )
# END SOLUTION: replace the ... below with the value named beside it

raw_text = freeform.content[0].text

print("What came back is a", type(raw_text).__name__)
print(raw_text)
print()

freeform_data = json.loads(raw_text)

print("Fields we got:     ", sorted(freeform_data.keys()))
print("Fields we expected:", sorted(REQUIRED_FIELDS))
print()
print("Expected but missing:", [f for f in REQUIRED_FIELDS if f not in freeform_data])
print("Not asked for:       ", [k for k in freeform_data if k not in REQUIRED_FIELDS])

check("freeform_json", freeform=freeform)

---

## Task 2 - Extract with a tool and a schema

In [ ]:
# ============================================================
# TASK 2 - Extract with a tool and a schema
# ============================================================
#
# WHAT TO DO
#   Send the same message again, but this time hand Claude a
#   tool whose input schema spells out every field, and force it
#   to use that tool.
#
# WHY IT MATTERS
#   Claude no longer writes JSON as text. It fills in a
#   structure you defined, and what comes back is already a
#   Python dictionary - nothing to parse, nothing to hope for.
#
#   Read the four comments inside the schema below. They are the
#   design choices that make a schema strict AND usable: fields
#   that are required but may be null, enums instead of free
#   text, an "unclear" option, and an "other" option with a
#   place to explain.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Now let's define the structure as a tool" - about
#   1 minute 42 seconds in.
#
# HOW TO DO IT
#   Two blanks in the request:
#
#       tools              the tools list defined just above
#       tool_choice name   "extract_return_request", in quotes -
#                          the tool Claude must use
# ============================================================

tools = [
    {
        "name": "extract_return_request",
        "description": "Extract a structured return request from a customer support message.",
        "input_schema": {
            "type": "object",
            "properties": {
                # (1) Required, but allowed to be null. Required means the field
                #     must EXIST. Nullable means the value may be missing. So
                #     Claude can say "no order id" instead of inventing one.
                "order_id": {
                    "type": ["string", "null"],
                    "description": "The customer's order ID, or null if not provided.",
                },
                "item": {
                    "type": ["string", "null"],
                    "description": "The item the customer wants to return or replace.",
                },
                # (2) An enum. Claude must pick one of these - no "product issue",
                #     no "return problem", nothing our router has no branch for.
                # (3) "unclear" is one of the options, so an ambiguous message can
                #     be labelled honestly instead of forced into a bad guess.
                # (4) "other" pairs with reason_detail below, so a case that fits
                #     nothing can still be described.
                "reason": {
                    "type": "string",
                    "enum": ["normal_return", "damaged_item", "billing_dispute",
                             "policy_exception", "unclear", "other"],
                    "description": "The main reason for the request.",
                },
                "reason_detail": {
                    "type": ["string", "null"],
                    "description": "Extra detail when reason is other or unclear.",
                },
                "desired_action": {
                    "type": "string",
                    "enum": ["refund", "replacement", "exchange", "store_credit",
                             "unclear", "other"],
                },
                "desired_action_detail": {
                    "type": ["string", "null"],
                    "description": "Extra detail when desired_action is other or unclear.",
                },
                "evidence_provided": {
                    "type": "boolean",
                    "description": "Whether the customer already provided evidence, such as a photo.",
                },
                "urgency": {
                    "type": "string",
                    "enum": ["low", "normal", "high", "unclear"],
                },
                "missing_information": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Information needed before the request can be processed.",
                },
            },
            "required": REQUIRED_FIELDS,
        },
    }
]


def extract_return_request(customer_message):
    prompt = f"""
Extract the return request from this customer message.

Customer message:
{customer_message}
"""

    # BEGIN SOLUTION
    message = client.messages.create(
        model=model,
        max_tokens=500,
        tools=tools,
        tool_choice={"type": "tool", "name": "extract_return_request"},
        messages=[
            {"role": "user", "content": prompt},
        ],
    )
    # SCAFFOLD: message = client.messages.create(
    # SCAFFOLD:     model=model,
    # SCAFFOLD:     max_tokens=500,
    # SCAFFOLD:     tools=...,   # the tools list defined just above
    # SCAFFOLD:     tool_choice={"type": "tool", "name": ...},   # "extract_return_request" - the tool it must use
    # SCAFFOLD:     messages=[
    # SCAFFOLD:         {"role": "user", "content": prompt},
    # SCAFFOLD:     ],
    # SCAFFOLD: )
    # END SOLUTION: replace each ... below with the value named beside it

    # Provided: the reply carries blocks, and we want the tool_use one.
    tool_use = next(block for block in message.content if block.type == "tool_use")
    return tool_use.input


extracted = extract_return_request(CUSTOMER_MESSAGE)

print("What came back is a", type(extracted).__name__)
print(json.dumps(extracted, indent=2))
print()
print("Same message, Task 1 gave us:", sorted(freeform_data.keys()))
print("Same message, Task 2 gave us:", sorted(extracted.keys()))

check("tool_extraction", extracted=extracted)

---

## Task 3 - Route on the extracted fields

In [ ]:
# ============================================================
# TASK 3 - Route on the extracted fields
# ============================================================
#
# WHAT TO DO
#   Write the function that decides where each extracted request
#   goes next.
#
# WHY IT MATTERS
#   This is the answer to "so what?". A schema is not worth the
#   effort because the output looks tidier. It is worth it
#   because these four lines of plain Python can read named
#   fields and send a case to the right place - no parsing, no
#   guessing what the model called things this time.
#
#   Order of the checks matters. A missing order_id comes first:
#   there is nothing useful to do with a return request for an
#   order you cannot identify, whatever the reason says.
#
# WHERE TO SEE IT IN THE LECTURE
#   "We can save it to a database. We can route the request to
#   the right workflow" - about 3 minutes 29 seconds in.
#
# HOW TO DO IT
#   Three blanks.
#
#   The first is what a nullable field holds when the customer
#   never gave one. In Python that value is written None.
#
#   The other two are values from the reason enum, in quotes.
#   The enum has six:
#
#       "normal_return"      an ordinary change of mind
#       "damaged_item"       arrived broken or scratched
#       "billing_dispute"    charged wrongly - not a returns job
#       "policy_exception"   outside the rules, needs a person
#       "unclear"            the message did not say
#       "other"              none of the above
# ============================================================

# BEGIN SOLUTION
def route(data):
    if data["order_id"] is None:
        return "ask_for_order_id"

    if data["reason"] == "billing_dispute":
        return "billing_team"

    if data["reason"] == "policy_exception":
        return "human_escalation"

    return "returns_workflow"
# SCAFFOLD: def route(data):
# SCAFFOLD:     if data["order_id"] is ...:   # None - what a nullable field holds when it is empty
# SCAFFOLD:         return "ask_for_order_id"
# SCAFFOLD:
# SCAFFOLD:     if data["reason"] == ...:   # "billing_dispute" - charged wrongly, not a returns job
# SCAFFOLD:         return "billing_team"
# SCAFFOLD:
# SCAFFOLD:     if data["reason"] == ...:   # "policy_exception" - outside the rules, needs a person
# SCAFFOLD:         return "human_escalation"
# SCAFFOLD:
# SCAFFOLD:     return "returns_workflow"
# END SOLUTION: replace each ... below with the value named beside it


# Provided: this task reuses the extractor you completed in the previous task.
if "extracted" not in globals():
    raise ValueError(
        "Run the previous cell successfully first - this task calls "
        "extract_return_request() from it.")


# Provided: three more customers, extracted the same way and routed.
INBOX = [
    "I got my shoes yesterday and they are scratched. I want a replacement.",
    "Order ORD-12345678 arrived broken and I need a refund today. Photo attached.",
    "I was charged twice for order ORD-87654321 and I want my money back.",
]

for customer_message in INBOX:
    data = extract_return_request(customer_message)
    print("{:<22} {:<16} order_id {}".format(
        route(data), data["reason"], data["order_id"]))
    print("    ", customer_message[:70])

check("routing", route=route)

---

## Done

Put the two outputs side by side one last time.

Task 1 gave you `"reason": "damaged"` and a key called `requested_action`. Task 2 gave
you `"reason": "damaged_item"` - a value from your own enum - under the name your code
already expects, plus `missing_information` telling you exactly what to ask the customer
for next.

The second one is not better because it is prettier. It is better because Task 3 could
be written at all.

One honest limit before you move on. A schema controls the *shape* of the answer, not
its *meaning*. Claude can return a perfectly valid `"normal_return"` for a message that
clearly describes a damaged item, and no schema will catch that. Checking whether the
values make sense is a separate job, and it is the next lesson.